# Testing DFT in CKKS to study OpenFHE and CKKS implementation

In [1]:
import openfhe as fhe
import numpy as np

In [ ]:
class CKKSRunner:
    def __init__(self):
        self.scale_mod_size = 50     # Q
        self.batch_size = 8          # N/2
        self.params = fhe.CCParamsCKKSRNS()

    def generateDFTmatrix(self, x):
        N = len(x)

        n = np.arange(N)
        k = n.reshape((N, 1))

        W = np.exp(-2j * np.pi * k * n / N)

        return W

    def matrixVectMult(self, cc: fhe.CryptoContext, x, W):
        

    def naiveHDFT(self, x):
        N = len(x)

        self.params.SetMultiplicativeDepth(1)
        self.params.SetScalingModSize(50)
        self.params.SetBatchSize(N)
        self.params.SetCKKSDataType(fhe.CKKSDataType.COMPLEX)

        cc = fhe.GenCryptoContext(self.params)

        cc.Enable(fhe.PKESchemeFeature.PKE)
        cc.Enable(fhe.PKESchemeFeature.KEYSWITCH)

        keyPair = cc.KeyGen()

        x_pt = cc.MakeCKKSPackedPlaintext(x)

        x_ct = cc.Encrypt(keyPair.publicKey, x_pt)

        W = self.generateDFTmatrix(x)



In [3]:
def main():
    input = [1.0 + 3j, 5.0 + 0j, 3.0 + 0j, 4.0 + 0j]

    ckksRunner = CKKSRunner()

    ckksRunner.naiveHDFT(input)

main()

In [ ]:
import openfhe as fhe
import math
import cmath

def create_dft_matrix(N):
    """Gera a matriz de Vandermonde para a DFT em texto limpo."""
    matrix = []
    for k in range(N):
        row = []
        for n in range(N):
            # Termo e^(-j * 2 * pi * k * n / N)
            angle = -2 * math.pi * k * n / N
            value = cmath.rect(1, angle)
            row.append(value)
        matrix.append(row)
    return matrix

def main():
    # 1. Configuração dos Parâmetros do Esquema CKKS
    parameters = fhe.CCParamsCKKSRNS()

    # Nível de profundidade multiplicativa.
    # A DFT ingênua exige apenas 1 multiplicação (Vetor Criptografado * Matriz Plaintext)
    parameters.SetMultiplicativeDepth(1)
    parameters.SetScalingModSize(50)

    # Definindo a quantidade de slots (deve ser potência de 2)
    # Para o teste, usaremos um vetor pequeno de tamanho 4
    N = 4
    parameters.SetBatchSize(N)
    parameters.SetCKKSDataType(fhe.CKKSDataType.COMPLEX)

    # Criando o CryptoContext
    crypto_context = fhe.GenCryptoContext(parameters)
    crypto_context.Enable(fhe.PKESchemeFeature.PKE)
    crypto_context.Enable(fhe.PKESchemeFeature.LEVELEDSHE)
    crypto_context.Enable(fhe.PKESchemeFeature.ADVANCEDSHE)
    crypto_context.Enable(fhe.PKESchemeFeature.KEYSWITCH)

    # Gerando as chaves
    key_pair = crypto_context.KeyGen()
    crypto_context.EvalSumKeyGen(key_pair.secretKey)

    # 2. Dados de Entrada
    # Vetor de exemplo que queremos transformar (pode ser real ou complexo)
    input_data = [1.0 + 0j, 2.0 + 0j, 3.0 + 0j, 4.0 + 0j]
    print(f"Entrada original: {input_data}")

    # Encriptando o vetor de entrada
    ptxt_input = crypto_context.MakeCKKSPackedPlaintext(input_data)
    ctxt_input = crypto_context.Encrypt(key_pair.publicKey, ptxt_input)

    # 3. Execução da DFT Homomórfica (Ingênua)
    # Geramos a matriz de constantes da DFT
    dft_matrix = create_dft_matrix(N)

    # Lista para guardar o resultado de cada componente X[k]
    ctxt_dft_components = []

    for k in range(N):
        # Para cada linha 'k' da matriz, criamos um Plaintext do OpenFHE
        ptxt_row = crypto_context.MakeCKKSPackedPlaintext(dft_matrix[k])

        # Multiplicação homomórfica elemento por elemento: x[n] * W^(kn)
        # Nota: Multiplicar Ciphertext por Plaintext não consome muita profundidade
        ctxt_mul = crypto_context.EvalMult(ctxt_input, ptxt_row)

        # Soma todos os elementos do vetor resultante para obter X[k]
        # O método EvalSum faz rotações internas e somas automaticamente para o tamanho do lote
        ctxt_sum = crypto_context.EvalSum(ctxt_mul, N)

        ctxt_dft_components.append(ctxt_sum)

    # 4. Reconstrução do Vetor Resultado (Apenas para demonstração)
    # Na prática, extrairíamos o primeiro elemento de cada 'ctxt_sum'
    print("\n--- Desencriptando os Resultados ---")
    dft_output = []
    for k in range(N):
        result_ptxt = crypto_context.Decrypt(ctxt_dft_components[k], key_pair.secretKey)
        result_ptxt.SetLength(N)
        # O valor correto de X[k] estará na primeira posição [0] devido ao comportamento do EvalSum
        dft_output.append(complex(result_ptxt.GetCKKSPackedValue()[0]))

    # Formatação para exibição limpa
    dft_output_rounded = [complex(round(c.real, 4), round(c.imag, 4)) for c in dft_output]
    print(f"Resultado da DFT Homomórfica: {dft_output_rounded}")

    # Validação com a DFT matemática pura para comparação
    expected = [sum(input_data[n] * dft_matrix[k][n] for n in range(N)) for k in range(N)]
    expected_rounded = [complex(round(c.real, 4), round(c.imag, 4)) for c in expected]
    print(f"Resultado Esperado (Matemático): {expected_rounded}")

if __name__ == "__main__":
    main()

Entrada original: [(1+0j), (2+0j), (3+0j), (4+0j)]

--- Desencriptando os Resultados ---
Resultado da DFT Homomórfica: [(10+0j), (-2+0j), (-2+0j), (-2+0j)]
Resultado Esperado (Matemático): [(10+0j), (-2+2j), (-2-0j), (-2-2j)]
